# FHIR Condition Bronze-to-Silver Transformation

## Purpose

In this notebook, I transform raw FHIR Condition resources from the Bronze
layer into a structured and analytics-ready Silver Delta table.

### Source

`health_insurance.bronze.fhir_condition_raw`

### Target

`health_insurance.silver.fhir_condition`

### What I am doing in this transformation

- I infer the combined FHIR Condition schema using a Serverless-compatible approach.
- I parse the raw JSON stored in Bronze into structured Spark data.
- I extract Condition identifiers, clinical status, verification status, and coding.
- I safely handle optional or empty FHIR coding arrays.
- I parse Patient and Encounter references into usable identifiers.
- I convert FHIR date-time fields to Spark timestamps.
- I derive the duration of resolved conditions where both dates are available.
- I standardize selected categorical values.
- I preserve ingestion metadata and add Silver transformation metadata.

not enforcing formal data-quality rules in this notebook. Those rules
will be attached to the production Lakeflow transformation so validation
happens while the Silver dataset is being produced.




In [0]:
# defining the source and target tables used by this transformation.

CATALOG = "health_insurance"

SOURCE_TABLE = f"{CATALOG}.bronze.fhir_condition_raw"
TARGET_TABLE = f"{CATALOG}.silver.fhir_condition"

print("Source:", SOURCE_TABLE)
print("Target:", TARGET_TABLE)


In [0]:
# loading the Bronze FHIR Condition table and inspecting its current shape.

condition_bronze_df = spark.table(SOURCE_TABLE)

print(f"Rows: {condition_bronze_df.count():,}")
print(f"Columns: {len(condition_bronze_df.columns)}")

condition_bronze_df.printSchema()

display(condition_bronze_df.limit(5))


## Inferring the FHIR Condition JSON schema

Because I am using Databricks Serverless, I avoid Spark RDD APIs.

I infer the schema directly from the Bronze `raw_json` column with
`schema_of_json_agg`. I use the aggregate form because FHIR fields are optional
and can vary between Condition resources, so the resulting schema reflects the
combined structure currently present in Bronze.


In [0]:
# inferring a combined FHIR Condition schema without using RDD operations.

schema_result = spark.sql(f'''
    SELECT schema_of_json_agg(raw_json) AS condition_schema
    FROM {SOURCE_TABLE}
''').first()

condition_schema = schema_result["condition_schema"]

print("FHIR Condition schema:")
print(condition_schema)


In [0]:
# parsing each raw Condition JSON string into a structured Spark column.

from pyspark.sql import functions as F

condition_parsed_df = (
    condition_bronze_df
    .withColumn(
        "condition",
        F.from_json(
            F.col("raw_json"),
            condition_schema
        )
    )
)

condition_parsed_df.select("condition.*").printSchema()


## Extracting the core Condition attributes

I now select the Condition fields that are needed by the conformed Silver
model while temporarily retaining the nested coding structures that I still
need to flatten.

FHIR fields are optional, so I allow missing values to remain `NULL` rather
than rejecting records during structural transformation.


In [0]:
# extracting the core Condition attributes and nested coding structures.

condition_core_df = (
    condition_parsed_df
    .select(
        F.col("condition.id").alias("condition_id"),

        F.col("condition.clinicalStatus").alias("clinical_status_struct"),
        F.col("condition.verificationStatus").alias("verification_status_struct"),
        F.col("condition.code").alias("condition_code_struct"),

        F.col("condition.subject.reference").alias("patient_reference"),
        F.col("condition.encounter.reference").alias("encounter_reference"),

        F.col("condition.onsetDateTime").alias("onset_datetime_raw"),
        F.col("condition.abatementDateTime").alias("abatement_datetime_raw"),
        F.col("condition.recordedDate").alias("recorded_datetime_raw"),

        "_ingested_at",
        "_source_system",
        "_resource_type"
    )
)


## Extracting Condition status values safely

FHIR stores both clinical and verification status inside `coding` arrays.

I use null-safe `get(array, 0)` access instead of positional `element_at()`.
If a coding array is empty, Spark returns `NULL` instead of failing the entire
transformation.


In [0]:
# extracting clinical and verification status with null-safe array access.

condition_status_df = (
    condition_core_df

    .withColumn(
        "clinical_status",
        F.expr("get(clinical_status_struct.coding.code, 0)")
    )

    .withColumn(
        "verification_status",
        F.expr("get(verification_status_struct.coding.code, 0)")
    )
)


## Extracting the Condition coding safely

The FHIR Condition `code` element can contain one or more coding systems.

For the conformed Silver table, I keep the first available coding entry and
extract its code, display value, and code system. I use null-safe array access
so an empty coding array becomes `NULL` rather than causing a runtime failure.


In [0]:
# extracting the primary Condition coding with null-safe array access.

condition_coded_df = (
    condition_status_df

    .withColumn(
        "condition_code",
        F.expr("get(condition_code_struct.coding.code, 0)")
    )

    .withColumn(
        "condition_name",
        F.expr("get(condition_code_struct.coding.display, 0)")
    )

    .withColumn(
        "code_system",
        F.expr("get(condition_code_struct.coding.system, 0)")
    )
)


## Parsing Patient and Encounter references

FHIR stores relationships as reference strings such as:

`Patient/<id>`

and

`Encounter/<id>`

I extract the identifier portion only when the reference has the expected
FHIR prefix. Missing or unexpected references remain `NULL` instead of being
converted to empty strings.


In [0]:
# converting FHIR Patient and Encounter references into clean identifiers.

condition_refs_df = (
    condition_coded_df

    .withColumn(
        "patient_id",
        F.when(
            F.col("patient_reference").startswith("Patient/"),
            F.regexp_extract(
                F.col("patient_reference"),
                r"^Patient/(.+)$",
                1
            )
        ).otherwise(F.lit(None).cast("string"))
    )

    .withColumn(
        "encounter_id",
        F.when(
            F.col("encounter_reference").startswith("Encounter/"),
            F.regexp_extract(
                F.col("encounter_reference"),
                r"^Encounter/(.+)$",
                1
            )
        ).otherwise(F.lit(None).cast("string"))
    )
)


## Converting FHIR timestamps

I convert the raw FHIR date-time strings into Spark timestamp columns so they
can be used consistently for filtering, comparisons, duration calculations,
and downstream analytical models.


In [0]:
# converting the raw Condition date-time fields to Spark timestamps.

condition_typed_df = (
    condition_refs_df

    .withColumn(
        "onset_datetime",
        F.to_timestamp("onset_datetime_raw")
    )

    .withColumn(
        "abatement_datetime",
        F.to_timestamp("abatement_datetime_raw")
    )

    .withColumn(
        "recorded_datetime",
        F.to_timestamp("recorded_datetime_raw")
    )
)


## Deriving Condition duration

When both onset and abatement dates are available, I calculate the number of
days between them.

An ongoing Condition may not have an abatement date. In that case I preserve
the duration as `NULL` instead of inventing a value. Logical validation such as
an abatement date occurring before onset will be handled by Lakeflow quality
expectations later.


In [0]:
# deriving Condition duration only when both dates are available.

condition_enriched_df = (
    condition_typed_df

    .withColumn(
        "condition_duration_days",
        F.when(
            F.col("onset_datetime").isNotNull()
            & F.col("abatement_datetime").isNotNull(),
            F.datediff(
                F.to_date("abatement_datetime"),
                F.to_date("onset_datetime")
            )
        ).otherwise(F.lit(None).cast("int"))
    )
)


## Standardizing Condition attributes

I standardize selected categorical values so Silver contains a consistent
representation for downstream joins and analytics.

I am only standardizing the values here. I am not dropping or failing records;
formal acceptance rules will be applied as Lakeflow expectations during
productionization.


In [0]:
# standardizing the Condition status values.

condition_standardized_df = (
    condition_enriched_df

    .withColumn(
        "clinical_status",
        F.upper(F.trim("clinical_status"))
    )

    .withColumn(
        "verification_status",
        F.upper(F.trim("verification_status"))
    )
)


## Building the final Silver Condition dataset

I now remove temporary nested structures and raw helper fields and keep the
conformed attributes required by the Silver Condition entity.

I preserve Bronze ingestion metadata and add `_silver_transformed_at` so the
row can be traced through the Medallion architecture.


In [0]:
# building the final conformed Silver Condition dataset.

condition_silver_df = (
    condition_standardized_df

    .select(
        "condition_id",
        "patient_id",
        "encounter_id",

        "condition_code",
        "condition_name",
        "code_system",

        "clinical_status",
        "verification_status",

        "onset_datetime",
        "abatement_datetime",
        "recorded_datetime",
        "condition_duration_days",

        "_source_system",
        "_resource_type",
        "_ingested_at"
    )

    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp()
    )
)


In [0]:
#  inspecting the final Silver Condition schema and a small result sample.

condition_silver_df.printSchema()

display(
    condition_silver_df.limit(20)
)


## Profiling optional and relationship fields

Before persisting the table, I profile fields that are useful for understanding
the completeness of the incoming Condition data.

These checks are descriptive only. I do not remove records here; the findings
will help me decide which Lakeflow expectations should warn, drop, or fail
during the production pipeline stage.


In [0]:
# profiling missing identifiers, coding, and FHIR relationships.

condition_silver_df.select(
    F.sum(F.col("condition_id").isNull().cast("int")).alias("missing_condition_id"),
    F.sum(F.col("patient_id").isNull().cast("int")).alias("missing_patient_id"),
    F.sum(F.col("encounter_id").isNull().cast("int")).alias("missing_encounter_id"),
    F.sum(F.col("condition_code").isNull().cast("int")).alias("missing_condition_code"),
    F.sum(F.col("clinical_status").isNull().cast("int")).alias("missing_clinical_status"),
    F.sum(
        F.col("verification_status").isNull().cast("int")
    ).alias("missing_verification_status")
).show()


In [0]:
# reconciling Bronze and Silver row counts before persistence.

bronze_count = condition_bronze_df.count()
silver_count = condition_silver_df.count()

print(f"Bronze Conditions: {bronze_count:,}")
print(f"Silver Conditions: {silver_count:,}")
print(f"Difference: {bronze_count - silver_count:,}")


## Persisting the Silver table

At this development stage, I overwrite the Silver table so the notebook remains
repeatable while I validate the transformation logic.

When I productionize this transformation in Lakeflow, the execution pattern
will become incremental and quality expectations will be evaluated while
Silver is being produced. The parsing and conformance logic developed here is
intended to remain reusable.


In [0]:
# persisting the conformed Condition dataset as a Delta table in Unity Catalog.

(
    condition_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(TARGET_TABLE)
)

print(f"Created Silver table: {TARGET_TABLE}")


In [0]:
%sql
-- verifying the number of Condition records written to Silver.

SELECT COUNT(*) AS condition_count
FROM health_insurance.silver.fhir_condition;


In [0]:
%sql
-- reviewing Condition status combinations as a sanity check.

SELECT
    clinical_status,
    verification_status,
    COUNT(*) AS condition_count
FROM health_insurance.silver.fhir_condition
GROUP BY clinical_status, verification_status
ORDER BY condition_count DESC;


## Transformation Result

successfully transformed the raw FHIR Condition resources from Bronze into a
structured Silver Delta table.

### Source

`health_insurance.bronze.fhir_condition_raw`

### Target

`health_insurance.silver.fhir_condition`

### Transformations I applied

- I inferred the combined FHIR Condition schema using a Serverless-compatible
  DataFrame/SQL approach.
- I parsed `raw_json` into structured Spark data.
- I extracted Condition identifiers, coding, clinical status, and verification status.
- I used null-safe array access for optional FHIR coding arrays.
- I parsed Patient and Encounter references into clean identifiers.
- I converted FHIR date-time fields into Spark timestamps.
- I derived Condition duration when both onset and abatement dates were available.
- I standardized selected categorical fields.
- I preserved Bronze lineage metadata and added a Silver transformation timestamp.
- I profiled missing relationship and coding fields without filtering records.
- I reconciled Bronze and Silver row counts before persistence.

### Data-quality boundary

did not rejecting records in this notebook.

Formal rules such as required Condition IDs, required Patient references,
required Condition coding, supported status values, and logical onset/abatement
dates will be attached as Lakeflow expectations when I productionize the
Bronze-to-Silver pipeline.

### Architecture

SMART FHIR API  
↓  
Auto Loader-managed Bronze ingestion  
↓  
`health_insurance.bronze.fhir_condition_raw`  
↓  
FHIR parsing and conformance  
↓  
`health_insurance.silver.fhir_condition`  
↓  
Lakeflow expectations / validated Silver  
↓  
Gold analytical model



### Status

**FHIR Condition Bronze-to-Silver transformation: COMPLETE**
